# Generative Phi-4-mini fine tuning

Train on the forensic dataset with two branches, split seed 321, training seed 42, two epochs and sequence length 4096.


In [ ]:
from pathlib import Path
import csv, json, ast, random, gc, statistics
from collections import defaultdict, Counter

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SOURCE_CSV = ROOT / 'data/forensic_atomic.csv'


RUN_MODE = "full"
MAX_INTERPRETATIONS = 2
MAX_SEQ_LEN = 4096
TRAIN_EPOCHS = 2
SAVE_STEPS = 100
SEED = 42
SPLIT_SEED = 321
MODEL_ID = "microsoft/Phi-4-mini-instruct"


EXP = ROOT / 'models/phi4_mini/generation'
SPLITS = EXP / "splits"
OUTPUTS = EXP / "adapters"

RELATIONS = [
    "xIntent", "xNeed", "xAttr", "xEffect", "xReact", "xWant",
    "oEffect", "oReact", "oWant", "isAfter", "HasSubEvent", "isBefore",
    "HinderedBy", "Causes", "xReason", "isFilledBy"
]
FIELDS = ["crime_category", "crime_subcategory", "brief_context"] + RELATIONS
TAXONOMY = {
    "Violent Crimes": ["Assault", "Homicide", "Kidnapping", "Robbery", "Sexual Offense"],
    "Property Crimes": ["Theft", "Burglary", "Arson", "Vandalism", "Trespassing"],
    "Financial Crimes": ["Fraud", "Money Laundering", "Embezzlement", "Bribery", "Tax Evasion"],
    "Cyber Crimes": ["Hacking", "Phishing", "Identity Theft", "Online Harassment"],
    "Organized Crimes": ["Drug Trafficking", "Human Trafficking", "Racketeering", "Weapon Smuggling"],
    "Public Order/Justice": ["Perjury", "Obstruction of Justice", "Disorderly Conduct"],
}
assert RUN_MODE in {"inspect", "smoke", "full"}
assert MAX_INTERPRETATIONS in {1, 2}
print("Mode:", RUN_MODE)
print("Source:", SOURCE_CSV)
print("Experiment:", EXP)


## Original DeepSeek prompt


In [ ]:
from models.phi4_mini.generation.prompt import SYSTEM_PROMPT, TRAIN_SYSTEM_PROMPT, get_user_prompt, user_prompt, messages_for


## Build grouped splits and targets


In [ ]:
def norm_event(s):
    return " ".join(str(s).casefold().split())

def parse_relation(value, line, field):
    s = (value or "").strip()
    if not s:
        return []
    try:
        x = json.loads(s)
    except json.JSONDecodeError:
        try:
            x = ast.literal_eval(s)
        except Exception as err:
            raise ValueError(f"Row {line}, {field}: invalid list") from err
    if not isinstance(x, (list, tuple)) or not all(isinstance(a, str) for a in x):
        raise ValueError(f"Row {line}, {field}: must be a list of strings")
    return list(x)

def read_source():
    if not SOURCE_CSV.exists():
        raise FileNotFoundError(f"Dataset not found: {SOURCE_CSV}")
    grouped = defaultdict(list)
    display = {}
    n_rows = 0
    with SOURCE_CSV.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        missing = {"event", *FIELDS} - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")
        for line, row in enumerate(reader, 2):
            event = (row["event"] or "").strip()
            if not event:
                raise ValueError(f"Row {line}: empty event")
            branch = {k: (row[k] or "").strip() for k in FIELDS[:3]}
            if branch["crime_subcategory"] not in TAXONOMY.get(branch["crime_category"], []):
                raise ValueError(f"Row {line}: category pair outside taxonomy")
            if not branch["brief_context"]:
                raise ValueError(f"Row {line}: empty brief_context")
            for k in RELATIONS:
                branch[k] = parse_relation(row[k], line, k)
            key = norm_event(event)
            display.setdefault(key, event)
            grouped[key].append(branch)
            n_rows += 1
    events = []
    discarded = 0
    odd_fills = 0
    for key in sorted(grouped):
        choices = []
        seen = set()
        for b in sorted(grouped[key], key=lambda x: (x["crime_category"], x["crime_subcategory"], x["brief_context"])):
            pair = (b["crime_category"], b["crime_subcategory"])
            if pair not in seen:
                choices.append(b)
                seen.add(pair)
        selected = choices[:1]
        if MAX_INTERPRETATIONS == 2 and len(choices) > 1:
            second = next((b for b in choices[1:] if b["crime_category"] != selected[0]["crime_category"]), choices[1])
            selected.append(second)
        discarded += len(grouped[key]) - len(selected)
        odd_fills += sum(bool(b["isFilledBy"]) for b in selected if "___" not in display[key])
        events.append({"event": display[key], "interpretations": selected})
    print(f"Source rows: {n_rows} | unique events: {len(events)}")
    print(f"Interpretations excluded by the 1–2 branch selection: {discarded}")
    print(f"Selected rows with nonempty isFilledBy and no ___: {odd_fills} (annotations preserved unchanged)")
    print("Branch distribution:", dict(Counter(len(e["interpretations"]) for e in events)))
    return events

def write_jsonl(path, rows):
    with path.open("x", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

SPLITS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
paths = {name: SPLITS / (name + ".jsonl") for name in ["train", "val", "test"]}
if all(p.exists() for p in paths.values()):
    print("Reusing existing splits without overwriting:", SPLITS)
else:
    if any(p.exists() for p in paths.values()):
        raise RuntimeError("Partial splits found; use a new experiment directory")
    events = read_source()
    random.Random(SPLIT_SEED).shuffle(events)
    n = len(events)
    train_end, val_end = int(n * .70), int(n * .85)
    write_jsonl(paths["train"], events[:train_end])
    write_jsonl(paths["val"], events[train_end:val_end])
    write_jsonl(paths["test"], events[val_end:])
    print("Created new splits:", SPLITS)
train_rows, val_rows, test_rows = [read_jsonl(paths[k]) for k in ("train", "val", "test")]
keys = [{norm_event(x["event"]) for x in rows} for rows in (train_rows, val_rows, test_rows)]
assert not (keys[0] & keys[1] or keys[0] & keys[2] or keys[1] & keys[2]), "Split overlap!"
print("Train / validation / test (events):", len(train_rows), len(val_rows), len(test_rows))
print("Example target:", json.dumps(train_rows[0]["interpretations"], ensure_ascii=False)[:850])


## Tokenization and length checks


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_prompt(event):
    return tokenizer.apply_chat_template(messages_for(event), tokenize=False, add_generation_prompt=True)

def render_completion(row):
    return json.dumps(row["interpretations"], ensure_ascii=False, separators=(",", ":")) + "<|end|>"

def preflight(rows):
    long = []
    counts = []
    for row in rows:

        n = len(tokenizer(render_prompt(row["event"]) + render_completion(row),
                          add_special_tokens=False)["input_ids"])
        counts.append(n)
        if n > MAX_SEQ_LEN:
            long.append((row["event"][:85], n))
    print(f"Preflight: {len(rows)} examples | max tokens: {max(counts, default=0)} | above {MAX_SEQ_LEN}: {len(long)}")
    if long:
        print("First overlong examples:", long[:5])
        raise ValueError("Some targets exceed MAX_SEQ_LEN; training with truncated JSON is blocked. "
                         "For a new experiment, use one interpretation or increase the limit if VRAM permits.")

print("Example prompt:\n", render_prompt(train_rows[0]["event"])[:1250])
print("Example response:\n", render_completion(train_rows[0])[:900])
if RUN_MODE == "inspect":
    print("Preflight complete for training and validation")
    preflight(train_rows)
    preflight(val_rows)


## QLoRA training


In [ ]:
def complete_checkpoints(folder):
    good = []
    for p in folder.glob("checkpoint-*"):
        if p.is_dir() and p.name.rsplit("-", 1)[-1].isdigit():
            if all((p / f).exists() for f in ("adapter_model.safetensors", "optimizer.pt", "scheduler.pt", "trainer_state.json")):
                good.append(p)
    return sorted(good, key=lambda p: int(p.name.rsplit("-", 1)[-1]))

def train_generation(full):
    folder = OUTPUTS / ("adapter_full" if full else "adapter_smoke")
    if (folder / "training_completed.json").exists() and (folder / "adapter_model.safetensors").exists():
        print("Adapter already complete; training skipped:", folder)
        return folder

    train = train_rows if full else train_rows[:64]
    val = val_rows if full else val_rows[:16]

    preflight(train)
    preflight(val)
    folder.mkdir(parents=True, exist_ok=True)
    checkpoints = complete_checkpoints(folder)
    resume = str(checkpoints[-1]) if checkpoints else None
    if resume:
        print("Resuming from checkpoint:", resume)

    d_train = Dataset.from_list([{"prompt": render_prompt(r["event"]), "completion": render_completion(r)} for r in train])
    d_val = Dataset.from_list([{"prompt": render_prompt(r["event"]), "completion": render_completion(r)} for r in val])

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU kernel before smoke or full training")
    use_bf16 = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    qconfig = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    model, trainer = None, None
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=qconfig,
                torch_dtype=dtype, trust_remote_code=True, device_map={"": torch.cuda.current_device()})
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(model)
        lora = LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32,
                          lora_dropout=0.05, target_modules="all-linear")
        args = SFTConfig(
            output_dir=str(folder), max_length=MAX_SEQ_LEN, completion_only_loss=True,
            num_train_epochs=TRAIN_EPOCHS if full else 1,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            gradient_accumulation_steps=8, learning_rate=2e-4,
            lr_scheduler_type="cosine", warmup_ratio=0.03,
            logging_strategy="steps", logging_steps=10 if full else 1,
            logging_first_step=True, save_strategy="steps", save_steps=SAVE_STEPS if full else 5,
            save_total_limit=3, eval_strategy="epoch", bf16=use_bf16, fp16=not use_bf16,
            gradient_checkpointing=True, packing=False, report_to="none", seed=SEED,
        )
        trainer = SFTTrainer(model=model, args=args, train_dataset=d_train, eval_dataset=d_val,
                             processing_class=tokenizer, peft_config=lora)
        trainer.train(resume_from_checkpoint=resume)
        trainer.save_model(str(folder))
        tokenizer.save_pretrained(str(folder))

        training_info = {
            "model": MODEL_ID, "seed": SEED, "split_seed": SPLIT_SEED,
            "epochs": TRAIN_EPOCHS if full else 1, "max_seq_len": MAX_SEQ_LEN,
            "max_interpretations": MAX_INTERPRETATIONS, "lora_r": 16,
            "lora_alpha": 32, "learning_rate": 2e-4,
            "train_examples": len(train), "val_examples": len(val),
            "optimizer_steps": trainer.state.global_step,
            "prompt_type": "original_prompts_py_SYSTEM_PROMPT_and_get_user_prompt",
        }
        (folder / "training_completed.json").write_text(json.dumps(training_info, indent=2), encoding="utf-8")
        (folder / "training_system_prompt.txt").write_text(TRAIN_SYSTEM_PROMPT, encoding="utf-8")
        print("Fine tuning complete:", folder)
    finally:
        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return folder


## Run mode


In [ ]:
if RUN_MODE == "inspect":
    print("Initial checks complete; training was not started.")
elif RUN_MODE == "smoke":
    train_generation(full=False)
elif RUN_MODE == "full":
    train_generation(full=True)
